# Data Enrichment

This notebook enriches the acquired homestay dataset by:

1. Generating price information.
2. Calculating distances from major tourist attractions.
3. Creating proximity labels.
4. Generating synthetic homestay descriptions using Qwen.

The enriched dataset will be used for feature engineering and recommendation generation.

In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

# Data manipulation
import pandas as pd
import numpy as np

from geopy.distance import geodesic

from ollama import chat

from tqdm import tqdm

In [2]:
# ==========================================
# LOAD ACQUIRED DATASET
# ==========================================

df = pd.read_csv(
    "../data/processed/homestays_acquired.csv"
)

print(df.shape)

df.head()

(1157, 15)


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile,google_name,google_address,latitude,longitude,rating,review_count
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,KALIMPONG,Municipality,"8th Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780,Revere Homsetay,"3F76+75W, Rishi Rd, Mongbol Busty, Kalimpong, ...",27.062761,88.460383,3.8,4
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,KALIMPONG,Municipality,"Chandralok, Kalimpong",santabgurung53@gmail.com,9932234895,Mansarover Homestay / Flora & Transport,"Kalimpong, West Bengal, Chandraloke, Kalimpong...",27.052709,88.461699,4.2,78
2,3,BETHANY HOMESTAY,ANUPAMA TAMANG,Silver,KALIMPONG,Kalimpong 1,Dr.GRAHAMS HOME BLOCK B,wangchuck20199@gmial.com,8348993048,Bethany Homestay Kalimpong,"Unnamed Road, Deolo, Dalapchan Ridge Reserve F...",27.082570,88.509402,4.6,70
3,4,S3 HOMESTAY,SANGITA RAI,Silver,KALIMPONG,Kalimpong 1,UPPER ECHHEY DARA GAON \r\nKALIMPONG,sangitasankalp@gmail.com,9933410313,Sunrise Inn Homestay,"E Main Rd, Chandraloke, Kalimpong, West Bengal...",27.053167,88.467557,4.3,118
4,5,BAJARANGI HOMESTAY,KAMAL KUMAR SHARMA,Silver,KALIMPONG,Kalimpong 1,SINGI SAMALBONG KALIMPONG,bajrangihomestay@gmail.com,8670450557,Bajrangi Homestay,"Sinji, Sukrabarey Bazar, Kalimpong, Samalbong ...",27.008221,88.512633,4.3,56


In [3]:
# ==========================================
# TOURIST LOCATIONS
# ==========================================

TOURIST_LOCATIONS = {
    "deolo": (27.08928, 88.50333), 
    "durpin": (27.03609, 88.45346), 
    "town": (27.05959, 88.46943), 
    "lava": (27.15992, 88.60598), 
    "pedong": (27.15945, 88.61551), 
    "gorubathan": (26.95435, 88.69557), 
    "rishop": (27.11166, 88.65187), 
    "lolegaon": (27.01909, 88.56538) 
}

In [4]:
# ==========================================
# DISTANCE CALCULATION FUNCTION
# ==========================================

def distance_km(
    lat,
    lon,
    target_lat,
    target_lon
):

    if pd.isna(lat) or pd.isna(lon):
        return None

    return round(
        geodesic(
            (lat, lon),
            (target_lat, target_lon)
        ).km,
        2
    )

In [5]:
# ==========================================
# DISTANCE FEATURE GENERATION
# ==========================================

for location_name, (target_lat, target_lon) in TOURIST_LOCATIONS.items():

    column_name = f"distance_to_{location_name}"

    df[column_name] = df.apply(
        lambda row: distance_km(
            row["latitude"],
            row["longitude"],
            target_lat,
            target_lon
        ),
        axis=1
    )

print("Distance features generated successfully.")

Distance features generated successfully.


In [6]:
# ==========================================
# PROXIMITY LABELS
# ==========================================

def proximity(distance):

    if pd.isna(distance):
        return None

    elif distance <= 5:
        return "Very Close"

    elif distance <= 10:
        return "Moderately Close"

    else:
        return "Far"

In [7]:
# ==========================================
# PROXIMITY FEATURE GENERATION
# ==========================================

df["deolo_proximity"] = (
    df["distance_to_deolo"]
    .apply(proximity)
)

df["durpin_proximity"] = (
    df["distance_to_durpin"]
    .apply(proximity)
)

df["town_proximity"] = (
    df["distance_to_town"]
    .apply(proximity)
)

In [8]:
# ==========================================
# PRICE GENERATION
# ==========================================

np.random.seed(42)

def generate_price(row):

    if str(row["category"]).upper() == "GOLD":
        return np.random.randint(2500, 4501)

    return np.random.randint(1000, 2201)

df["price"] = df.apply(
    generate_price,
    axis=1
)

df["price"].describe()

count    1157.000000
mean     1663.660328
std       469.431350
min      1000.000000
25%      1325.000000
50%      1667.000000
75%      1935.000000
max      4498.000000
Name: price, dtype: float64

In [10]:
# ==========================================
# AMENITIES FEATURE GENERATION
# ==========================================

def assign_amenities(row):

    if row["category"] == "Gold":

        return pd.Series({
            "wifi": np.random.choice([0,1], p=[0.1,0.9]),
            "parking": np.random.choice([0,1], p=[0.15,0.85]),
            "breakfast": np.random.choice([0,1], p=[0.1,0.9]),
            "mountain_view": np.random.choice([0,1], p=[0.3,0.7]),
            "room_service": np.random.choice([0,1], p=[0.4,0.6]),
            "bonfire_barbeque": np.random.choice([0,1], p=[0.5,0.5]),
            "pickup_dropoff_service": np.random.choice([0,1], p=[0.6,0.4])
        })

    else:

        return pd.Series({
            "wifi": np.random.choice([0,1], p=[0.3,0.7]),
            "parking": np.random.choice([0,1], p=[0.25,0.75]),
            "breakfast": np.random.choice([0,1], p=[0.2,0.8]),
            "mountain_view": np.random.choice([0,1], p=[0.4,0.6]),
            "room_service": np.random.choice([0,1], p=[0.75,0.25]),
            "bonfire_barbeque": np.random.choice([0,1], p=[0.6,0.4]),
            "pickup_dropoff_service": np.random.choice([0,1], p=[0.8,0.2])
        })

df[[
    "wifi",
    "parking",
    "breakfast",
    "mountain_view",
    "room_service",
    "bonfire_barbeque",
    "pickup_dropoff_service"
]] = df.apply(assign_amenities, axis=1)

In [11]:
# ==========================================
# LOAD PRE-GENERATED DESCRIPTIONS
# ==========================================

df_desc = pd.read_csv(
    "../data/processed/homestays_generated_description.csv"
)

df["description"] = df_desc["description"]

In [12]:
# ==========================================
# SAVE ENRICHED DATASET
# ==========================================

df.to_csv(
    "../data/processed/homestays_enriched.csv",
    index=False
)

print(
    "Enriched dataset saved successfully."
)

Enriched dataset saved successfully.
